In [34]:
import numpy as np
import pandas as pd
import re
from sentence_transformers import SentenceTransformer

## Models:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import classification_report, confusion_matrix

In [2]:
data = pd.read_csv("synthetic_sensitive_prompt_dataset.csv")

In [3]:
data

,id,text,label
0,SYN-001,Can you summarise this public article about re...,safe
1,SYN-002,Explain the difference between random forests ...,safe
2,SYN-003,Write a polite email asking my lecturer for cl...,safe
3,SYN-004,Give me Python code to plot a sine wave using ...,safe
4,SYN-005,What are the main causes of inflation in simpl...,safe
...,...,...,...
113,SYN-114,"This deployment log is noisy, but I think the ...",credential
114,SYN-115,I am practising for a data science interview. ...,safe
115,SYN-116,Please turn this onboarding note into a checkl...,pii
116,SYN-117,The finance note says the customer paid invoic...,financial


In [ ]:
def find_common_regex_patterns(df, text_col = "text"):
    # Phone, Email, address, credit card, api key, password ,postcode
    patterns = {
        "has_email": r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
        "has_phone": r"(\+44\s?7\d{3}|\(?07\d{3}\)?)\s?\d{3}\s?\d{3}",
        "has_credit_card": r"\b(?:\d[ -]*?){13,16}\b",
        "has_api_key": r"\b(sk_live|sk_test|ghp|AKIA|xoxb|whsec)_[A-Za-z0-9_\-]{8,}\b",
        "has_password": r"(?i)(password|passwd|pwd)\s*[:=]\s*\S+",
        "has_postcode": r"\b[A-Z]{1,2}\d[A-Z\d]?\s?\d[A-Z]{2}\b"
    }

    for feature, pattern in patterns.items():
        df[feature] = df[text_col].apply(lambda x:1 if re.search(pattern, str(x)) else 0)
    return df

def embedding_output(df):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(df["text"].tolist())
    return embeddings
def classifier(X, y):
    clf = LogisticRegression(max_iter = 1000)
    clf.fit(X, y)
    return clf


def encode_labels(df, label_col="label"):
    encoder = LabelEncoder()
    y = encoder.fit_transform(df[label_col])

    label_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))

    return y, encoder, label_mapping

def make_stratified_split(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    return X_train, X_test, y_train, y_test


def evaluate_model(y_true, y_pred, encoder=None):
    if encoder is not None:
        class_names = encoder.classes_
    else:
        class_names = None

    print("Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)

    if class_names is not None:
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    else:
        cm_df = pd.DataFrame(cm)

    print("\nConfusion Matrix:")
    print(cm_df)

    return cm_df



In [ ]:
data = find_common_regex_patterns(data)
embeddings = embedding_output(data)
embeddings_df = pd.DataFrame(embeddings, columns = [f"embed_{i}" for i in range(embeddings.shape[1])])

final_df = pd.concat([data, embeddings_df], axis = 1)
drop_cols = ["id", "text"]
final_df = final_df.drop(columns = drop_cols)
X = final_df.drop(columns = ["label"])
y, encoder, label_mapping = encode_labels(final_df, label_col="label")
# print(label_mapping)
X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=1,
        stratify=y
    )
# print(X_train)
model = classifier(X_train, y_train)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

     has_email  has_phone  has_credit_card  has_api_key  has_password  \
13           0          0                0            0             0   
31           0          0                0            0             0   
21           0          1                0            0             0   
90           0          0                0            0             0   
83           0          0                0            0             0   
..         ...        ...              ...          ...           ...   
25           1          1                0            0             0   
12           0          0                0            0             0   
108          0          1                0            0             0   
62           0          0                0            0             0   
93           0          0                0            0             0   

     has_postcode   embed_0   embed_1   embed_2   embed_3  ...  embed_374  \
13              0 -0.026882  0.086039  0.01715

In [38]:
####### THIS IS ON TEST DATA:
y_pred = model.predict(X_test)

cm_df = evaluate_model(y_test, y_pred, encoder)

Classification Report:
                       precision    recall  f1-score   support

confidential_business       0.33      1.00      0.50         3
           credential       0.75      1.00      0.86         3
            financial       1.00      0.50      0.67         2
               health       0.00      0.00      0.00         1
             legal_hr       0.00      0.00      0.00         2
      mixed_sensitive       0.00      0.00      0.00         2
                  pii       0.50      0.67      0.57         3
                 safe       1.00      0.80      0.89         5
          source_code       1.00      0.67      0.80         3

             accuracy                           0.62        24
            macro avg       0.51      0.51      0.48        24
         weighted avg       0.61      0.62      0.58        24


Confusion Matrix:
                       confidential_business  credential  financial  health  \
confidential_business                      3           0 

c:\Users\Hamza Sajjad\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Hamza Sajjad\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Hamza Sajjad\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [14]:
data

,id,text,label,has_email,has_phone,has_credit_card,has_api_key,has_password,has_postcode
0,SYN-001,Can you summarise this public article about re...,safe,0,0,0,0,0,0
1,SYN-002,Explain the difference between random forests ...,safe,0,0,0,0,0,0
2,SYN-003,Write a polite email asking my lecturer for cl...,safe,0,0,0,0,0,0
3,SYN-004,Give me Python code to plot a sine wave using ...,safe,0,0,0,0,0,0
4,SYN-005,What are the main causes of inflation in simpl...,safe,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
113,SYN-114,"This deployment log is noisy, but I think the ...",credential,0,0,0,0,1,0
114,SYN-115,I am practising for a data science interview. ...,safe,0,0,0,0,0,0
115,SYN-116,Please turn this onboarding note into a checkl...,pii,1,1,0,0,0,0
116,SYN-117,The finance note says the customer paid invoic...,financial,0,0,0,0,0,0


In [24]:
type(embeddings)

numpy.ndarray